# ANALYSE-12 : Compter les couplages parfaits — le permanent comme pont, Lean d'un côté, énumération de l'autre

Pli 6 de l'Epic Origami `openai/math` (#19898, tranche B : #21958). La série ANALYSE confronte un énoncé de recherche à ce que le dépôt sait prouver ; ici, l'énoncé est un problème de **comptage**.

- **Trouver** un couplage parfait dans un graphe biparti : polynomial (flot, assignment hongrois — le lac `assignment_lean` l'exhibe, cf. `Duality.lean`).
- **Compter** les couplages parfaits : la classe `#P` — et l'objet matriciel qui porte ce comptage est le **permanent**, cousin signe-ignorant du déterminant.

La tranche A (#21962) a déposé dans le lac le pont `card_eq_permanent` : le permanent (organe natif Mathlib) de la matrice d'adjacence 0/1 **égale** le cardinal des couplages parfaits. Ce carnet fait trois choses :

1. il audite le socle (l'organe Mathlib `Matrix.permanent`, le fichier `Counting.lean`) et vérifie les signatures et les axiomes depuis le lac ;
2. il recoupe le pont par l'énumération brute en Python — chaque valeur que Lean évalue doit sortir identique de l'énumération ;
3. il exerce les contrôles négatifs natifs (`Matrix.permanent_zero`, `Matrix.permanent_one`) comme garants de sens.

**Statut épistémique d'ensemble** (doctrine #19898, détaillé §7) : ce dépôt prouve le pont local ; la `#P`-complétude (Valiant) et le FPRAS (Jerrum–Sinclair–Vigoda) sont des théorèmes classiques **cités** ; le squelette `lean/docs` publié par les auteurs d'`openai/math` (famille 113) est **rapporté, non vérifié ici**.

**Environnement d'exécution** : kernel `python3` (le venv du dépôt) ; les cellules Lean compilent des sondes via `wsl` + `lake env lean` sur le lac `assignment_lean` (toolchain v4.33.0, pin Mathlib `db584cd6d46c`) — convention de la série `-Lean-Python` (cf. ANALYSE-06).

## 1. L'énoncé — compter les couplages parfaits (famille 113)

Un graphe biparti $G = (U \cup V, E)$ avec $|U| = |V| = n$, représenté par sa matrice d'adjacence booléenne $B$ ($B_{ij}$ : l'arête $i \to j$ existe).

- Un **couplage parfait** est une bijection $\sigma : U \to V$ telle que toutes les arêtes $(i, \sigma(i))$ existent.
- Le **permanent** d'une matrice $M$ carrée $n \times n$ est $\mathrm{perm}(M) = \sum_{\sigma \in S_n} \prod_{i=1}^{n} M_{i,\sigma(i)}$ — la somme sur **toutes** les permutations, comme le déterminant **sans les signes**.

Sur une matrice 0/1, chaque produit vaut $1$ exactement quand $\sigma$ est un couplage couvert : $\mathrm{perm}(B) = \#\{\text{couplages parfaits}\}$. Compter, c'est déjà évaluer le permanent.

**Le saut de classe** : l'existence d'un couplage parfait se décide en temps polynomial ; compter les couplages parfaits d'un graphe biparti est `#P`-complet — c'est le problème **fondateur** de la classe (Valiant, 1979), et il admet néanmoins un FPRAS randomisé (Jerrum–Sinclair–Vigoda, 2004). Ces deux théorèmes sont des classiques de la littérature, cités ici comme tels ; le statut du squelette formel publié par les auteurs d'`openai/math` est traité au §7.

## 2. La chaîne de la preuve — trois maillons, un seul à nous

| Maillon | Porteur | Statut dans ce carnet |
|---|---|---|
| 1. L'organe `Matrix.permanent` existe (zéro, un, transposée, diagonal…) | Mathlib au pin `v4.33.0` | **vérifié** (audit §3 + sondes) |
| 2. Le pont `card_eq_permanent` : permanent de l'adjacence = cardinal des couplages | le lac `assignment_lean` (tranche A, #21962) | **vérifié** (signatures + axiomes + `#eval` recoupés §5) |
| 3. `#P`-complétude du comptage ; FPRAS ; frontière SDP (famille 126) | littérature classique / squelette `openai/math` | **cité / rapporté** — jamais « prouvé par nous » |

La méthode du carnet suit la série : ne jamais confondre ce que le dépôt **prouve** (maillons 1-2, avec `#print axioms` à l'appui) et ce qu'il **cite** (maillon 3). Le recoupement Python du §4-§5 ne prouve pas le pont — il vérifie que la formalisation ne s'est pas trompée d'objet : chaque valeur Lean doit sortir identique de l'énumération brute.

## 3. Le socle Lean — audit de l'organe et du pont

Deux vérifications préalables, faites **à la source** : l'organe `Matrix.permanent` est bien dans Mathlib au pin du lac (avec ses contrôles natifs `permanent_zero`, `permanent_one`), et le fichier `Counting.lean` de la tranche A est présent dans le lac avec les déclarations attendues.

In [1]:
# Audit Mathlib au pin v4.33.0 (db584cd6d46c) : l'organe permanent et ses contrôles natifs.
import subprocess, os

LAKE_DIR = (subprocess.run(["wsl", "wslpath", "-a", os.getcwd().replace("\\", "/")],
            capture_output=True, text=True, encoding="utf-8", errors="replace")
            .stdout.strip() + "/../../../GameTheory/assignment_lean")
MATHLIB = LAKE_DIR + "/.lake/packages/mathlib"

def wsl_sh(cmd):
    return subprocess.run(["wsl", "-e", "bash", "-lc", cmd],
                          capture_output=True, text=True,
                          encoding="utf-8", errors="replace").stdout.strip()

print("toolchain du lac :", wsl_sh("cat " + LAKE_DIR + "/lean-toolchain"))
print()
print("organe Mathlib — Mathlib/LinearAlgebra/Matrix/Permanent.lean :")
print(" ", wsl_sh("stat -c %s " + MATHLIB + "/Mathlib/LinearAlgebra/Matrix/Permanent.lean"), "octets")
print()
print("théorèmes natifs `permanent_*` (extraits) :")
for t in wsl_sh("grep -hE '^theorem permanent_' " + MATHLIB
                + "/Mathlib/LinearAlgebra/Matrix/Permanent.lean | sed 's/ :=.*//' | head -6").splitlines():
    print("  ", t.strip())

toolchain du lac : leanprover/lean4:v4.33.0

organe Mathlib — Mathlib/LinearAlgebra/Matrix/Permanent.lean :
  4370 octets

théorèmes natifs `permanent_*` (extraits) :


   theorem permanent_diagonal {d : n → R} : permanent (diagonal d) = ∏ i, d i
   theorem permanent_zero [Nonempty n] : permanent (0 : Matrix n n R) = 0
   theorem permanent_one : permanent (1 : Matrix n n R) = 1
   theorem permanent_isEmpty [IsEmpty n] {A : Matrix n n R} : permanent A = 1
   theorem permanent_eq_one_of_card_eq_zero {A : Matrix n n R} (h : card n = 0) : permanent A = 1
   theorem permanent_unique {n : Type*} [Unique n] [DecidableEq n] [Fintype n] (A : Matrix n n R) :


### Lecture de l'audit

L'organe existe au pin : `Permanent.lean` porte la définition et une famille de théorèmes natifs dont les deux contrôles utilisés au §6 (`permanent_zero`, `permanent_one`) et la transposée utilisée par la preuve du pont (`permanent_transpose`). Le lac tourne sous la toolchain `v4.33.0` — le pin sous lequel la tranche A a été prouvée.

**Organ-first** (#13564) : le lac ne redéfinit ni le permanent ni ses lemmes — il invoque l'organe Mathlib et prouve la correspondance avec les couplages.

In [2]:
# Lecture du socle : Counting.lean (tranche A, #21962) — extraits numérotés.
COUNTING_WIN = os.path.join(os.getcwd(), "..", "..", "..",
                            "GameTheory", "assignment_lean", "Assignment", "Counting.lean")
lines = open(COUNTING_WIN, encoding="utf-8").read().splitlines()
print("fichier :", os.path.relpath(COUNTING_WIN, os.getcwd()), f"({len(lines)} lignes)")
print()
for a, b in [(23, 32), (50, 57), (74, 80)]:
    print(f"--- lignes {a}-{b} ---")
    for k in range(a - 1, b):
        print(f"{k + 1:3d} | {lines[k]}")
    print()

fichier : ..\..\..\GameTheory\assignment_lean\Assignment\Counting.lean (82 lignes)

--- lignes 23-32 ---
 23 | /-- Matrice d'adjacence entiere d'un graphe biparti sur `Fin n × Fin n` :
 24 | `1` si l'arete existe, `0` sinon. -/
 25 | def adjMatrix (B : Matrix (Fin n) (Fin n) Bool) : Matrix (Fin n) (Fin n) ℕ :=
 26 |   B.map fun b => if b then 1 else 0
 27 | 
 28 | /-- L'ensemble des couplages parfaits d'un graphe biparti : les permutations
 29 | dont toutes les aretes ligne `i` vers colonne `σ i` sont presentes. -/
 30 | def perfectMatchingSet (B : Matrix (Fin n) (Fin n) Bool) :
 31 |     Finset (Equiv.Perm (Fin n)) :=
 32 |   Finset.univ.filter fun σ => ∀ i, B i (σ i)

--- lignes 50-57 ---
 50 | /-- **Le pont** : le permanent (Mathlib) de la matrice d'adjacence 0/1 egale
 51 | le nombre de couplages parfaits du graphe biparti. C'est la formalisation
 52 | locale du point d'entree des familles 113 (FPRAS pour compter) et 126
 53 | (cloture SDP) d'openai/math, citees avec leur statut ep

### Lecture du socle

Les deux définitions locales font le pont entre les graphes et l'algèbre : `adjMatrix` (0/1 ← Bool) et `perfectMatchingSet` (les permutations couvertes, comme `Finset`). L'énoncé `card_eq_permanent` dit exactement l'égalité annoncée ; le corollaire `permanent_complete_eq_factorial` la confronte au factoriel sur le graphe complet — témoin croisé avec `Fintype.card_perm`. La preuve (visible dans le fichier) réécrit le filtre-cardinal en somme d'indicateurs puis reconnaît la définition du permanent, en s'appuyant sur `Matrix.permanent_transpose` — aucun calcul caché.

In [3]:
# Vérification formelle depuis le lac : signatures + axiomes (lake env lean).
probe1 = '''import Assignment.Counting

#check @Assignment.card_eq_permanent
#check @Assignment.perfectMatchingSet
#check @Assignment.adjMatrix
#check @Matrix.permanent_zero
#check @Matrix.permanent_one
#check @Matrix.permanent_transpose
#print axioms Assignment.card_eq_permanent
#print axioms Matrix.permanent_transpose
'''
probe_name = "_analyse12_probe1.lean"
try:
    with open(probe_name, "w", encoding="utf-8", newline="\n") as f:
        f.write(probe1)
    probe_wsl = subprocess.run(["wsl", "wslpath", "-a",
                                (os.getcwd() + os.sep + probe_name).replace("\\", "/")],
                               capture_output=True, text=True, encoding="utf-8",
                               errors="replace").stdout.strip()
    r = subprocess.run(["wsl", "-e", "bash", "-lc",
                        "cd " + LAKE_DIR + " && timeout 850 lake env lean " + probe_wsl + " 2>&1"],
                       capture_output=True, text=True, encoding="utf-8",
                       errors="replace", timeout=900)
    print(r.stdout)
    print("rc =", r.returncode)
finally:
    if os.path.exists(probe_name):
        os.remove(probe_name)

@Assignment.card_eq_permanent : ∀ {n : ℕ} (B : Matrix (Fin n) (Fin n) Bool),
  (Assignment.perfectMatchingSet B).card = (Assignment.adjMatrix B).permanent
@Assignment.perfectMatchingSet : {n : ℕ} → Matrix (Fin n) (Fin n) Bool → Finset (Equiv.Perm (Fin n))
@Assignment.adjMatrix : {n : ℕ} → Matrix (Fin n) (Fin n) Bool → Matrix (Fin n) (Fin n) ℕ
@Matrix.permanent_zero : ∀ {n : Type u_1} [inst : DecidableEq n] [inst_1 : Fintype n] {R : Type u_2}
  [inst_2 : CommSemiring R] [Nonempty n], Matrix.permanent 0 = 0
@Matrix.permanent_one : ∀ {n : Type u_1} [inst : DecidableEq n] [inst_1 : Fintype n] {R : Type u_2}
  [inst_2 : CommSemiring R], Matrix.permanent 1 = 1
@Matrix.permanent_transpose : ∀ {n : Type u_1} [inst : DecidableEq n] [inst_1 : Fintype n] {R : Type u_2}
  [inst_2 : CommSemiring R] (M : Matrix n n R), M.transpose.permanent = M.permanent
'Assignment.card_eq_permanent' depends on axioms: [propext, Classical.choice, Quot.sound]
'Matrix.permanent_transpose' depends on axioms: [propext,

### Lecture des `#check` et `#print axioms`

Les signatures sont celles attendues : `card_eq_permanent` parle bien du cardinal du `Finset` des permutations couvertes contre le permanent **de l'adjacence** (une matrice `ℕ`), et les deux contrôles natifs de Mathlib sont des énoncés **sans argument** (`permanent 0 = 0`, `permanent 1 = 1`).

Les axiomes : `card_eq_permanent` ne dépend que des trois axiomes standard de Mathlib (`propext`, `Classical.choice`, `Quot.sound`) — **ni `sorryAx`** (pas de trou), **ni `Lean.ofReduceBool`** (pas de décision native sans preuve). C'est le témoin d'intégrité formelle du maillon 2.

## 4. Le permanent par énumération brute — le recoupement Python

Le chemin Python est volontairement **naïf et indépendant** du lac : `permanent_def` applique la définition (somme sur `itertools.permutations` de produits), `count_matchings` compte directement les couplages parfaits (test booléen des arêtes). Deux chemins de code disjoints qui doivent rendre le même nombre — puis confrontation **exhaustive** sur tout l'espace des matrices 0/1 pour $n \le 3$.

In [4]:
# Le permanent par définition, le comptage direct des couplages, et une galerie de témoins.
from itertools import permutations
import math

def permanent_def(M):
    n = len(M)
    return sum(math.prod(M[i][s[i]] for i in range(n)) for s in permutations(range(n)))

def count_matchings(M):
    n = len(M)
    return sum(1 for s in permutations(range(n)) if all(M[i][s[i]] for i in range(n)))

def edges_to_matrix(n, edges):
    M = [[0] * n for _ in range(n)]
    for i, j in edges:
        M[i][j] = 1
    return M

K33 = edges_to_matrix(3, [(i, j) for i in range(3) for j in range(3)])
B3 = edges_to_matrix(3, [(0, 0), (0, 2), (1, 0), (1, 1), (2, 1), (2, 2)])
BLOC4 = edges_to_matrix(4, [(0, 0), (0, 1), (1, 0), (1, 1), (2, 2), (2, 3), (3, 2), (3, 3)])
CYC4 = edges_to_matrix(4, [(i, i) for i in range(4)] + [(i, (i + 1) % 4) for i in range(4)])
VIDE3 = edges_to_matrix(3, [])
ID3 = edges_to_matrix(3, [(i, i) for i in range(3)])

print(f"{'graphe témoin':<24}{'permanent_def':>14}{'count_matchings':>17}")
print("-" * 57)
for nom, M in [("K3,3 (complet)", K33), ("B3 (6 arêtes)", B3),
               ("bloc K2,2 + K2,2", BLOC4), ("cycle C4", CYC4),
               ("vide (3x3)", VIDE3), ("identité I3", ID3)]:
    p, c = permanent_def(M), count_matchings(M)
    print(f"{nom:<24}{p:>14}{c:>17}   {'accord' if p == c else 'ÉCART'}")
print()
print("K3,3 =", math.factorial(3), "— le factoriel prédit par permanent_complete_eq_factorial (lignes 74-80 ci-dessus).")

graphe témoin            permanent_def  count_matchings
---------------------------------------------------------
K3,3 (complet)                       6                6   accord
B3 (6 arêtes)                        2                2   accord
bloc K2,2 + K2,2                     4                4   accord
cycle C4                             2                2   accord
vide (3x3)                           0                0   accord
identité I3                          1                1   accord

K3,3 = 6 — le factoriel prédit par permanent_complete_eq_factorial (lignes 74-80 ci-dessus).


### Lecture du résultat

Les deux chemins Python s'accordent sur toute la galerie, et les valeurs illustrent la théorie : le graphe complet rend $3! = 6$ (le corollaire du lac), le graphe **vide** rend $0$ (aucun couplage — c'est le contenu de `permanent_zero` au §6), l'identité rend $1$ (`permanent_one`), et le bloc $K_{2,2} \oplus K_{2,2}$ rend $2 \times 2 = 4$ : le comptage est **multiplicatif sur l'union disjointe** — propriété qu'un simple «finder» polynomial ne demande pas.

In [5]:
# Confrontation exhaustive : TOUTES les matrices 0/1 pour n <= 3, plus la symétrie
# par transposée (pendant Python de Matrix.permanent_transpose).
ecarts = 0
total = 0
for n in [1, 2, 3]:
    for bits in range(2 ** (n * n)):
        M = [[(bits >> (i * n + j)) & 1 for j in range(n)] for i in range(n)]
        total += 1
        if permanent_def(M) != count_matchings(M):
            ecarts += 1
        if permanent_def(M) != permanent_def([list(r) for r in zip(*M)]):
            ecarts += 1
print(f"{total} matrices balayées (espace 0/1 complet pour n <= 3, transposées incluses) : {ecarts} écart(s)")

530 matrices balayées (espace 0/1 complet pour n <= 3, transposées incluses) : 0 écart(s)


### Lecture de la confrontation

Sur l'espace **complet** des petites instances (toute matrice 0/1 jusqu'à $n = 3$, soit l'enumeration exhaustive et non un échantillon), l'égalité permanent = comptage ne vacille jamais, et la symétrie par transposée tient — le pendant calculatoire de `Matrix.permanent_transpose` que la preuve du pont utilise. Ce n'est pas une preuve du pont (c'est le rôle du maillon 2), mais c'est la détection d'erreur d'objet : la formalisation compte bien ce qu'on croit.

## 5. Le pont confronté — `#eval` Lean contre énumération Python

Le test décisif : Lean **évalue** (`#eval`) le cardinal du `perfectMatchingSet` et le permanent de l'adjacence sur les témoins concrets, puis Python relit ces valeurs et les compare à sa propre énumération. Les deux matrices littérales (`B3`, `BLOC4`) sont écrites une seule fois côté Lean — le notebook impose qu'elles décrivent les mêmes graphes que les listes d'arêtes du §4.

In [6]:
# Le pont mesuré des deux côtés : #eval Lean vs énumération Python (B3, BLOC4).
B3_LEAN = "!![true, false, true; true, true, false; false, true, true]"
BLOC4_LEAN = ("!![true, true, false, false; "
              "true, true, false, false; "
              "false, false, true, true; "
              "false, false, true, true]")
probe2 = f'''import Assignment.Counting
open Assignment
def B3L : Matrix (Fin 3) (Fin 3) Bool := {B3_LEAN}
def BLOC4L : Matrix (Fin 4) (Fin 4) Bool := {BLOC4_LEAN}
#eval (perfectMatchingSet B3L).card
#eval (adjMatrix B3L).permanent
#eval (perfectMatchingSet BLOC4L).card
#eval (adjMatrix BLOC4L).permanent
'''
probe_name = "_analyse12_probe2.lean"
try:
    with open(probe_name, "w", encoding="utf-8", newline="\n") as f:
        f.write(probe2)
    probe_wsl = subprocess.run(["wsl", "wslpath", "-a",
                                (os.getcwd() + os.sep + probe_name).replace("\\", "/")],
                               capture_output=True, text=True, encoding="utf-8",
                               errors="replace").stdout.strip()
    r = subprocess.run(["wsl", "-e", "bash", "-lc",
                        "cd " + LAKE_DIR + " && timeout 850 lake env lean " + probe_wsl + " 2>&1"],
                       capture_output=True, text=True, encoding="utf-8",
                       errors="replace", timeout=900)
    print(r.stdout)
    lean_vals = [int(l) for l in r.stdout.splitlines() if l.strip().isdigit()]
finally:
    if os.path.exists(probe_name):
        os.remove(probe_name)

py_vals = [count_matchings(B3), permanent_def(B3), count_matchings(BLOC4), permanent_def(BLOC4)]
noms = ["card(perfectMatchingSet B3)", "permanent(adjMatrix B3)",
        "card(perfectMatchingSet BLOC4)", "permanent(adjMatrix BLOC4)"]
print(f"{'quantité':<34}{'Lean #eval':>12}{'Python':>9}   verdict")
accords = 0
for nom, lv, pv in zip(noms, lean_vals, py_vals):
    ok = lv == pv
    accords += ok
    print(f"{nom:<34}{lv:>12}{pv:>9}   {'accord' if ok else 'ÉCART'}")
print()
print(f"pont card_eq_permanent recoupé sur les témoins : {accords}/4 accords")

2
2
4
4

quantité                            Lean #eval   Python   verdict
card(perfectMatchingSet B3)                  2        2   accord
permanent(adjMatrix B3)                      2        2   accord
card(perfectMatchingSet BLOC4)               4        4   accord
permanent(adjMatrix BLOC4)                   4        4   accord

pont card_eq_permanent recoupé sur les témoins : 4/4 accords


### Lecture du pont mesuré

Les quatre quantités s'accordent : le cardinal **calculé par le noyau Lean** sur la définition ensembliste du lac et le permanent **calculé par la définition Mathlib** rendent exactement les nombres de l'énumération Python. Le pont n'est pas une coïncidence d'énoncé — les deux côtés calculent la même chose sur des implémentations indépendantes (noyau Lean contre interpréteur Python).

## 6. Les contrôles négatifs natifs — `permanent_zero` et `permanent_one`

Mathlib fournit lui-même les deux garants de sens du comptage : une matrice **nulle** a un permanent **nul** (aucun couplage dans un graphe sans arêtes), et la matrice **identité** a un permanent **égal à 1** (exactement un couplage parfait : la permutation identité). Ce sont des théorèmes **démontrés** dans l'organe — le carnet les re-déclare comme `example` (preuve : le lemme natif, sans argument) et les confronte aux mêmes graphes côté Python.

In [7]:
# Contrôles négatifs natifs Mathlib + contre-épreuve Python (graphe vide, identité).
probe3 = '''import Assignment.Counting
open Matrix Assignment
#check @Matrix.permanent_zero
#check @Matrix.permanent_one
example : (0 : Matrix (Fin 3) (Fin 3) ℕ).permanent = 0 := Matrix.permanent_zero
example : (1 : Matrix (Fin 3) (Fin 3) ℕ).permanent = 1 := Matrix.permanent_one
#eval (adjMatrix (!![false, false, false; false, false, false; false, false, false])).permanent
#eval (adjMatrix (!![true, false, false; false, true, false; false, false, true])).permanent
#eval (perfectMatchingSet (!![false, false, false; false, false, false; false, false, false])).card
#eval (perfectMatchingSet (!![true, false, false; false, true, false; false, false, true])).card
#print axioms Matrix.permanent_zero
#print axioms Matrix.permanent_one
'''
probe_name = "_analyse12_probe3.lean"
try:
    with open(probe_name, "w", encoding="utf-8", newline="\n") as f:
        f.write(probe3)
    probe_wsl = subprocess.run(["wsl", "wslpath", "-a",
                                (os.getcwd() + os.sep + probe_name).replace("\\", "/")],
                               capture_output=True, text=True, encoding="utf-8",
                               errors="replace").stdout.strip()
    r = subprocess.run(["wsl", "-e", "bash", "-lc",
                        "cd " + LAKE_DIR + " && timeout 850 lake env lean " + probe_wsl + " 2>&1"],
                       capture_output=True, text=True, encoding="utf-8",
                       errors="replace", timeout=900)
    print(r.stdout)
finally:
    if os.path.exists(probe_name):
        os.remove(probe_name)

print("contre-épreuve Python : permanent_def(VIDE3) =", permanent_def(VIDE3),
      "| count_matchings(VIDE3) =", count_matchings(VIDE3),
      "| permanent_def(ID3) =", permanent_def(ID3),
      "| count_matchings(ID3) =", count_matchings(ID3))

@permanent_zero : ∀ {n : Type u_1} [inst : DecidableEq n] [inst_1 : Fintype n] {R : Type u_2} [inst_2 : CommSemiring R]
  [Nonempty n], permanent 0 = 0
@permanent_one : ∀ {n : Type u_1} [inst : DecidableEq n] [inst_1 : Fintype n] {R : Type u_2} [inst_2 : CommSemiring R],
  permanent 1 = 1
0
1
0
1
'Matrix.permanent_zero' depends on axioms: [propext, Classical.choice, Quot.sound]
'Matrix.permanent_one' depends on axioms: [propext, Classical.choice, Quot.sound]

contre-épreuve Python : permanent_def(VIDE3) = 0 | count_matchings(VIDE3) = 0 | permanent_def(ID3) = 1 | count_matchings(ID3) = 1


### Lecture des contrôles

Les `example` compilent : les deux énoncés natifs s'appliquent **tels quels** aux tailles concrètes (aucun argument à fournir — ce sont des égalités closes). Les `#eval` rendent `0` et `1` comme attendu, et la contre-épreuve Python sur les mêmes graphes (`VIDE3`, `ID3` du §4) donne les mêmes nombres. Les axiomes des deux lemmes natifs sont les trois standards — les contrôles du comptage ne reposent sur rien de plus fort que le pont lui-même.

## 7. Statut épistémique par famille — 113, 126, 104

Doctrine de l'Epic #19898 : chaque affirmation héritée d'`openai/math` porte son statut, **par famille**. Jamais « OpenAI a prouvé X donc X ».

### Famille 113 — compter les couplages parfaits (FPRAS, `#P`)

- **Littérature classique, citée** : la `#P`-complétude du comptage des couplages parfaits bipartis (Valiant, 1979) et l'existence d'un FPRAS randomisé (Jerrum–Sinclair–Vigoda, 2004) sont des théorèmes publiés et peer-reviewés, cités ici au niveau de leur énoncé — ce dépôt ne les re-prouve pas.
- **`openai/math` (rapporté)** : la famille 113 du dépôt des auteurs publie un squelette `lean/docs` pour ce volet comptage. Statut : **rapporté** — squelette publié par les auteurs, non vérifié par ce dépôt.
- **Ce dépôt prouve** : le pont local `card_eq_permanent` (maillon 2, axiomes standards, §3) — la porte d'entrée formelle du comptage, pas les bornes de complexité.

### Famille 126 — clôture SDP du polytope du couplage

- Résultat **négatif** (aucune remontée semidefinie compacte) ; même grille de lecture : énoncé cité de la littérature, squelette `openai/math` **rapporté**, rien de vérifié localement au-delà du pont qui définit l'objet compté.

### Famille 104 — en veille

- Aucun traitement dans ce pli ; mentionnée pour mémoire du paysage.

## 8. Friction et chemin de découverte

Trois frictions réelles rencontrées en écrivant ce carnet, conservées parce qu'elles enseignent la méthode :

1. **Les signatures se sondent, elles ne se devinent pas.** La première rédaction du contrôle `example` appliquait un argument à `Matrix.permanent_zero` (`permanent_zero _`) — une sonde `#check` a révélé que l'énoncé natif est **close** (`permanent 0 = 0`, sans quantificateur sur la matrice). Un cycle de sonde vaut mieux qu'un cycle d'exécution de carnet raté.
2. **Le coût d'entrée du lac est l'import Mathlib.** Chaque sonde `lake env lean` paie l'import (mesuré : ~5 min par sonde via le montage WSL /mnt/d, secondes sur un lac en système de fichiers natif) — les sondes sont donc **regroupées** en trois fichiers plutôt qu'éparpillées par cellule, avec un délai d'attente dimensionné pour l'import froid (850 s).
3. **Deux littéraux pour un même graphe.** Les témoins existent côté Lean (littéraux `!![...]`) et côté Python (listes d'arêtes) ; le carnet impose la discipline de les écrire **côte à côte dans la même cellule** pour que la désynchronisation reste visible.

## 9. Exercices

Trois exercices, du calcul au sens. Convention de la série : stubs sans erreur volontaire — le notebook s'exécute de bout en bout même non complété.

In [8]:
# Exercice 1 -- Ryser : le permanent par inclusion-exclusion (a completer)
# But : evaluer le permanent SANS enumerer les n! permutations -- par la formule de
# Ryser : perm(M) = (-1)^n * sum_{S sous-ensemble non vide des colonnes} (-1)^{|S|} * prod_i (sum_{j in S} M[i][j])
# Etape 1 : implementer ryser(M) avec itertools.combinations (ou un balayage de bits).
# Etape 2 : verifier que ryser coincide avec permanent_def sur TOUS les temoins du
# paragraphe 4 (K33, B3, BLOC4, CYC4, VIDE3, ID3).
# Indice : pour n <= 6, le balayage des 2^n - 1 sous-ensembles est immediat.
def ryser(M):
    # TODO etudiant
    return None

print("Exercice a completer")

Exercice a completer


### Exercice 2 — Union de deux cycles : prédire, puis mesurer

Un graphe biparti 2-régulier est une union de cycles pairs ; chaque cycle $C_{2k}$ porte exactement deux couplages parfaits. **Avant de mesurer** : combien de couplages parfaits pour l'union de deux $C_4$ ($n = 8$) ? Puis construisez la matrice et vérifiez.

In [9]:
# Exercice 2 -- union de deux C4 (n=8) : PREDIRE le compte, puis MESURER (a completer)
# Etape 1 : construire la matrice 8x8 de l'union de deux cycles C4 :
# cycle 1 sur les sommets 0-3 (colonnes 0-3), cycle 2 sur les sommets 4-7 (colonnes 4-7),
# chaque cycle reliant i a i et (i+1) mod 4 dans son bloc.
# Etape 2 : noter la prediction ici (en commentaire), puis mesurer avec count_matchings
# et permanent_def -- les trois valeurs doivent coherir.
def union_deux_c4():
    # TODO etudiant
    return None

print("Exercice a completer")

Exercice a completer


### Exercice 3 — La limite du pont : entrées négatives

Le pont `card_eq_permanent` vit sur les matrices d'adjacence 0/1. Que devient l'interprétation comptage quand le permanent s'évalue sur une matrice à coefficients arbitraires — par exemple une entrée $-1$ ? Mesurez, puis expliquez pourquoi la preuve du pont (produits d'indicateurs) ne s'étend pas.

In [10]:
# Exercice 3 -- une entree negative : le permanent compte-t-il encore ? (a completer)
M_neg = [[1, 1], [-1, 1]]
# Etape 1 : evaluer permanent_def(M_neg) et decrire le resultat (signe, valeur).
# Etape 2 : en une phrase (en commentaire), expliquer pourquoi ce nombre n'est PLUS
# un compte de couplages -- quel maillon de la preuve du pont (produits d'indicateurs 0/1)
# casse avec une entree negative ?
print("Exercice a completer")

Exercice a completer


## 10. Conclusion

- **Ce que le dépôt sait maintenant** : le permanent Mathlib de la matrice d'adjacence égale le cardinal des couplages parfaits — pont **prouvé** (tranche A), **vérifié** ici par signatures, axiomes et `#eval` recoupés contre l'énumération Python exhaustive sur les petites instances.
- **Ce que ce carnet cite sans le prouver** : la `#P`-complétude (Valiant), le FPRAS (JSV), la frontière SDP (famille 126) — statut par famille au §7 ; le squelette `openai/math` de la famille 113 reste **rapporté**.
- **Ce que compter a de plus que trouver** : la galerie du §4 le montre sur un exemple — le comptage est multiplicatif sur l'union disjointe, il distingue des graphes également « couvrables », et c'est précisément cette richesse qui le place dans `#P` alors que l'existence reste polynomiale.

Le maillon local est fermé ; la frontière de complexité reste un énoncé cité — et c'est exactement la distinction que la série ANALYSE existe pour tenir.